# Načtení a příprava dat

In [ ]:
# nacteni vstupnich dat
# The pickle ships with the repo at the project root:
#   Data_lindat_zipformer_ft2_lm-extra06_allFeatures.pkl

import pandas as pd
import pickle
with open("Data_lindat_zipformer_ft2_lm-extra06_allFeatures.pkl", "rb") as f:
    raw_data = pickle.load(f)

# rychle prohlednuti
raw_data

,kobar_kategorizace_definitivni,kobar_kategorizace_podle_RBANS,kobar_kategorizace_podle_ALBAV,kobar_kategorizace_podle_MAST,osoba_age,osoba_educationYears,osoba_sex,rbans_vyhodnoceni_souhrnny_skor,rbans_bezprostredni_pamet_test_uceni,rbans_bezprostredni_pamet_test_pameti,...,allFeatures_t10_tfidf_char_'ó',allFeatures_t10_tfidf_char_'ý',allFeatures_t10_tfidf_char_'č',allFeatures_t10_tfidf_char_'ň',allFeatures_t10_tfidf_char_'š',allFeatures_t10_mean_lev_f1_score,session_id,screening_id,splits,is_valid
0,0,0,0.0,0.0,60,20,1,121,34,21,...,NaN,NaN,NaN,NaN,NaN,NaN,i2W6KnmrYCBstJcR2YKNdm,scr-UFYw7EWtGytm93w4PcbnPY,train,True
1,0,0,0.0,0.0,75,18,0,126,33,16,...,NaN,NaN,NaN,NaN,NaN,NaN,dY2fr8RRPkSUoUdLYvYtbu,scr-NMnNHjnnGBrmwA7gWzJTNp,train,True
2,0,0,0.0,1.0,51,12,0,104,28,13,...,NaN,NaN,NaN,NaN,NaN,NaN,midcyThzXN6CUuKB5WmBZK,scr-Rbhnpx9Ms9qnCt43KkkSJB,train,True
3,0,0,0.0,1.0,69,19,1,104,26,20,...,NaN,NaN,NaN,NaN,NaN,NaN,C2iTC82ScZMsxfwSyT6FBy,scr-SPnEaj2zeULNDiCvLuaXhN,train,True
4,0,0,0.0,0.0,62,17,1,99,25,18,...,NaN,NaN,NaN,NaN,NaN,NaN,hM5GeZ9iUgs8f2FnfxxPHs,scr-LbyqVJeyu4zBfpmwz4hCKF,test,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
460,0,0,0.0,NaN,74,13,0,123,34,23,...,0.0,0.0,0.183155,0.0,0.0,0.7964,6a2fa3ee260933714ff61fac,scr-6a2fa2bf260933714ff61f94,train.extra,True
461,3,2,1.0,NaN,73,13,0,61,15,9,...,0.0,0.0,0.158877,0.0,0.0,0.6642,6a1d742a260933714ff400ec,scr-6a1d7480260933714ff40124,train.extra,True
462,0,0,0.0,NaN,76,20,1,122,26,15,...,0.0,0.0,0.208783,0.0,0.0,0.6880,6a157fa07d3e8142b3c1ff9f,scr-6a157e727d3e8142b3c1ff82,train.extra,True
463,0,0,0.0,NaN,53,12,0,119,30,19,...,0.0,0.0,0.120811,0.0,0.0,0.5688,6a146a147d3e8142b3c1e91e,scr-6a1457207d3e8142b3c1e6d2,train.extra,True


In [ ]:
# kolik raw zaznamu mame...
raw_data.shape

(465, 3222)

In [ ]:
# pocty v jednotlivych splitech, train extra idealne nechat pak na experimenty s objevovanim LLM features, resp. ty ktere maji jako diagnozu 0, 2, 3 (bez 1, ta bude vyfiltrovana nasledne)
raw_data['splits'].value_counts()

,count
splits,
train,294
train.extra,94
test,77


In [ ]:
# distribuce diagnoz, nas budou zajimat vsechny krome hodnoty 1

diagnosis_distribution = raw_data["kobar_kategorizace_definitivni"].value_counts()
display(diagnosis_distribution)

,count
kobar_kategorizace_definitivni,
0,267
1,77
2,64
3,57


In [ ]:
df_train = raw_data[raw_data["splits"].isin(["train", "train.extra"])]

for diag in [0, 1, 2, 3]:
    count = len(df_train[df_train["kobar_kategorizace_definitivni"] == diag])
    print(f"Diagnóza {diag}: {count}")

Diagnóza 0: 225
Diagnóza 1: 61
Diagnóza 2: 54
Diagnóza 3: 48


In [ ]:
# jen inspekce...
raw_data.columns

Index(['kobar_kategorizace_definitivni', 'kobar_kategorizace_podle_RBANS',
       'kobar_kategorizace_podle_ALBAV', 'kobar_kategorizace_podle_MAST',
       'osoba_age', 'osoba_educationYears', 'osoba_sex',
       'rbans_vyhodnoceni_souhrnny_skor',
       'rbans_bezprostredni_pamet_test_uceni',
       'rbans_bezprostredni_pamet_test_pameti',
       ...
       'allFeatures_t10_tfidf_char_'ó'', 'allFeatures_t10_tfidf_char_'ý'',
       'allFeatures_t10_tfidf_char_'č'', 'allFeatures_t10_tfidf_char_'ň'',
       'allFeatures_t10_tfidf_char_'š'', 'allFeatures_t10_mean_lev_f1_score',
       'session_id', 'screening_id', 'splits', 'is_valid'],
      dtype='object', length=3222)

In [ ]:
# filtr diagnozy: (0, 2, 3)
filtered_data = raw_data[raw_data["kobar_kategorizace_definitivni"].isin([0, 2, 3])].copy()

In [ ]:
# binarizace 'new_diagnosis_label' pres lambda funkci
filtered_data['new_diagnosis_label'] = filtered_data["kobar_kategorizace_definitivni"].apply(lambda x: 1 if x in [2, 3] else 0)

In [ ]:
# KLICOVA DATA PRO NAS: new_diag, task4 (popis obrazku), splits, nic dalsiho nechceme zpracovavat...
final_df = filtered_data[['new_diagnosis_label', 'task4_Complex scene description_obraz u jezera_recognized', 'splits']].copy()
display(final_df.head())

,new_diagnosis_label,task4_Complex scene description_obraz u jezera_recognized,splits
0,0,Kachna s kachňaty plave. Slečna si čte knížku ...,train
1,0,Žena se leží v lehátku a čte knihu s červeným ...,train
2,0,Pán loví ryby. Pes honí veverku. Děti hází míč...,train
3,0,Dáma v lehátku čte knihu. Děti si hrajou s míč...,train
4,0,Kluk si háže míč s holčičkou. Letadlo letí na ...,test


In [ ]:
# rozdeleni na train a test df, inspekce

train_df = final_df[final_df['splits'].isin(['train', 'train.extra'])].drop(columns=['splits'])
test_df = final_df[final_df['splits'] == 'test'].drop(columns=['splits'])

print("Train DataFrame head:")
display(train_df.head())
print("\nTest DataFrame head:")
display(test_df.head())

Train DataFrame head:


,new_diagnosis_label,task4_Complex scene description_obraz u jezera_recognized
0,0,Kachna s kachňaty plave. Slečna si čte knížku ...
1,0,Žena se leží v lehátku a čte knihu s červeným ...
2,0,Pán loví ryby. Pes honí veverku. Děti hází míč...
3,0,Dáma v lehátku čte knihu. Děti si hrajou s míč...
5,0,Takže chlapeček s holčičkou si hrajou s balóne...



Test DataFrame head:


,new_diagnosis_label,task4_Complex scene description_obraz u jezera_recognized
4,0,Kluk si háže míč s holčičkou. Letadlo letí na ...
13,0,"V pozadí si hrajou děti, házejí si míčkem, běh..."
31,1,Tak paní leží pod slunečníkem. Čte knihu. Tady...
32,0,"Pes honí veverku, chlapec a dívka si hází míče..."
33,0,Paní si čte na lehátku knížku. Je pod sluneční...


# ML klasifikační část

In [ ]:
# importy a priprava
!pip install -q transformers datasets scikit-learn

import numpy as np
import torch
from torch.utils.data import TensorDataset, DataLoader
from torch.optim import AdamW
from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_linear_schedule_with_warmup
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, roc_auc_score
from tqdm.auto import tqdm
import random

In [ ]:
# parametry modelu, trenovani, ...

# testovací data -- doplnena/obohacena
OUTPUT_TEST_CSV_PATH = "./texttest_predicted.csv"

# Model, jeho nastavení a trénovací parametry
MODEL_NAME = "fav-kky/FERNET-C5-RoBERTa"
MAX_LENGTH = 256
BATCH_SIZE = 8
EPOCHS = 8
LEARNING_RATE = 1e-5
WARMUP_RATIO = 0.1

# Pro reproducibilitu
SEED = 1704
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# nutime na GPU, kontrola
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Použité zařízení:", device)

Použité zařízení: cuda


In [ ]:
# Nacteni tokenizeru a modelu pro binarni klasifikaci

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)
model.to(device)

config.json:   0%|          | 0.00/637 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.06k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/932k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/592k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.62M [00:00<?, ?B/s]

...

model.safetensors: reconstructing file:   0%|          |  0.00B /  498MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: fav-kky/FERNET-C5-RoBERTa
Key                        | Status     | 
---------------------------+------------+-
lm_head.dense.weight       | UNEXPECTED | 
lm_head.layer_norm.weight  | UNEXPECTED | 
lm_head.dense.bias         | UNEXPECTED | 
lm_head.bias               | UNEXPECTED | 
lm_head.layer_norm.bias    | UNEXPECTED | 
classifier.out_proj.weight | MISSING    | 
classifier.out_proj.bias   | MISSING    | 
classifier.dense.bias      | MISSING    | 
classifier.dense.weight    | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


RobertaForSequenceClassification(
  (roberta): RobertaModel(
    (embeddings): RobertaEmbeddings(
      (word_embeddings): Embedding(50000, 768, padding_idx=1)
      (token_type_embeddings): Embedding(1, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
      (position_embeddings): Embedding(514, 768, padding_idx=1)
    )
    (encoder): RobertaEncoder(
      (layer): ModuleList(
        (0-11): 12 x RobertaLayer(
          (attention): RobertaAttention(
            (self): RobertaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): RobertaSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
             

In [ ]:
# Klicove sloupce: new_diagnosis_label, task4_Complex scene description_obraz u jezera_recognized

required_columns = {"task4_Complex scene description_obraz u jezera_recognized", "new_diagnosis_label"}
if not required_columns.issubset(train_df.columns):
    raise ValueError(f"Trénovací data musí obsahovat sloupce: {required_columns}")

# Odstraneni radku s NaN v textu
train_df = train_df.dropna(subset=["task4_Complex scene description_obraz u jezera_recognized", "new_diagnosis_label"]).copy()

# Prevod na spravne typy
train_df["task4_Complex scene description_obraz u jezera_recognized"] = train_df["task4_Complex scene description_obraz u jezera_recognized"].astype(str)
train_df["new_diagnosis_label"] = train_df["new_diagnosis_label"].astype(int)

texts_train_all = train_df["task4_Complex scene description_obraz u jezera_recognized"].tolist()
labels_train_all = train_df["new_diagnosis_label"].values

print("Počet instancí v trénovacích datech:", len(texts_train_all))

Počet instancí v trénovacích datech: 327


In [ ]:
# Split to TRAIN/VAL

train_texts, val_texts, train_labels, val_labels = train_test_split(
    texts_train_all,
    labels_train_all,
    test_size=0.25,
    random_state=SEED,
    stratify=labels_train_all
)

print("Počet instancí v trénovací množině:", len(train_texts))
print("Počet instancí ve validační množině:", len(val_texts))

Počet instancí v trénovací množině: 245
Počet instancí ve validační množině: 82


In [ ]:
# funkce na tokenizaci...
def tokenize_texts(text_list, tokenizer, max_length):
    """
    Tokenizuje seznam textů *text_list* za pomoci *tokenizer* s určením max délky *max_length*: return tensory *input_ids* a *attention_mask*.
    """
    encodings = tokenizer(
        text_list,
        padding=True,
        truncation=True,
        max_length=max_length,
        return_tensors="pt"
    )
    input_ids = encodings["input_ids"]
    attention_mask = encodings["attention_mask"]
    return input_ids, attention_mask

In [ ]:
# Samotna tokenizace: train a val data

print("\nTokenizuji trénovací a validační data...")

train_input_ids, train_attention_mask = tokenize_texts(
    train_texts, tokenizer, MAX_LENGTH
)
val_input_ids, val_attention_mask = tokenize_texts(
    val_texts, tokenizer, MAX_LENGTH
)

train_labels_tensor = torch.tensor(train_labels, dtype=torch.long)
val_labels_tensor = torch.tensor(val_labels, dtype=torch.long)

# Vytvoření datasetu a dataloaderu
train_dataset = TensorDataset(
    train_input_ids, train_attention_mask, train_labels_tensor
)
val_dataset = TensorDataset(
    val_input_ids, val_attention_mask, val_labels_tensor
)

train_dataloader = DataLoader(
    train_dataset, batch_size=BATCH_SIZE, shuffle=True
)
val_dataloader = DataLoader(
    val_dataset, batch_size=BATCH_SIZE, shuffle=False
)


Tokenizuji trénovací a validační data...


In [ ]:
# Optimalizace + scheduler
optimizer = AdamW(model.parameters(), lr=LEARNING_RATE)
total_steps = len(train_dataloader) * EPOCHS
num_warmup_steps = int(WARMUP_RATIO * total_steps)
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=num_warmup_steps,
    num_training_steps=total_steps
)

In [ ]:
print("\n===== ZAČÁTEK TRÉNOVÁNÍ =====")

for epoch in range(EPOCHS):
    print(f"\n--- Epocha {epoch + 1}/{EPOCHS} ---")
    model.train()
    total_train_loss = 0.0

    for batch in tqdm(train_dataloader, desc="Trénování"):
        batch_input_ids = batch[0].to(device)
        batch_attention_mask = batch[1].to(device)
        batch_labels = batch[2].to(device)

        optimizer.zero_grad()

        outputs = model(
            input_ids=batch_input_ids,
            attention_mask=batch_attention_mask,
            labels=batch_labels
        )

        loss = outputs.loss
        loss.backward()

        # Ořezání gradientů (pro stabilitu) - napovězeno ChatGPT
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

        optimizer.step()
        scheduler.step()

        total_train_loss += loss.item()

    avg_train_loss = total_train_loss / len(train_dataloader)
    print(f"Průměrná trénovací ztráta: {avg_train_loss:.4f}")

    # =======================
    # EVALUACE NA VAL SUBSETU
    # =======================
    model.eval()
    total_val_loss = 0.0
    all_val_labels = []
    all_val_probs = []

    with torch.no_grad():
        for batch in tqdm(val_dataloader, desc="Validace"):
            batch_input_ids = batch[0].to(device)
            batch_attention_mask = batch[1].to(device)
            batch_labels = batch[2].to(device)

            outputs = model(
                input_ids=batch_input_ids,
                attention_mask=batch_attention_mask,
                labels=batch_labels
            )

            loss = outputs.loss
            logits = outputs.logits

            total_val_loss += loss.item()

            # Pravděpodobnosti tříd (softmax přes dvě třídy)
            probs = torch.softmax(logits, dim=1)[:, 1]  # pravděpodobnost třídy 1

            all_val_labels.extend(batch_labels.cpu().numpy())
            all_val_probs.extend(probs.cpu().numpy())

    avg_val_loss = total_val_loss / len(val_dataloader)
    all_val_labels = np.array(all_val_labels)
    all_val_probs = np.array(all_val_probs)
    val_preds = (all_val_probs >= 0.5).astype(int)

    val_acc = accuracy_score(all_val_labels, val_preds)
    val_precision, val_recall, val_f1, _ = precision_recall_fscore_support(
        all_val_labels,
        val_preds,
        average="binary",
        zero_division=0
    )

    try:
        val_auc = roc_auc_score(all_val_labels, all_val_probs)
    except ValueError:
        val_auc = float("nan")

    print(f"Validační ztráta: {avg_val_loss:.4f}")
    print(f"Validační Accuracy:  {val_acc:.4f}")
    print(f"Validační Precision: {val_precision:.4f}")
    print(f"Validační Recall:    {val_recall:.4f}")
    print(f"Validační F1:        {val_f1:.4f}")
    print(f"Validační ROC AUC:   {val_auc:.4f}")

print("\n===== TRÉNOVÁNÍ DOKONČENO =====")


===== ZAČÁTEK TRÉNOVÁNÍ =====

--- Epocha 1/8 ---


Trénování:   0%|          | 0/31 [00:00<?, ?it/s]

Průměrná trénovací ztráta: 0.6249


Validace:   0%|          | 0/11 [00:00<?, ?it/s]

Validační ztráta: 0.5741
Validační Accuracy:  0.6829
Validační Precision: 0.0000
Validační Recall:    0.0000
Validační F1:        0.0000
Validační ROC AUC:   0.7081

--- Epocha 2/8 ---


...

Validační ztráta: 0.6670
Validační Accuracy:  0.7683
Validační Precision: 0.6842
Validační Recall:    0.5000
Validační F1:        0.5778
Validační ROC AUC:   0.8510

--- Epocha 8/8 ---


Trénování:   0%|          | 0/31 [00:00<?, ?it/s]

Průměrná trénovací ztráta: 0.1819


Validace:   0%|          | 0/11 [00:00<?, ?it/s]

Validační ztráta: 0.6741
Validační Accuracy:  0.7683
Validační Precision: 0.6842
Validační Recall:    0.5000
Validační F1:        0.5778
Validační ROC AUC:   0.8475

===== TRÉNOVÁNÍ DOKONČENO =====


In [ ]:
# =========================================
# TEST
# =========================================

if not required_columns.issubset(test_df.columns):
    raise ValueError(
        f"Testovací data musí obsahovat sloupce: {required_columns} "
        f"(task4_Complex scene description_obraz u jezera_recognized a new_diagnosis_label)."
    )

test_df = test_df.dropna(subset=["task4_Complex scene description_obraz u jezera_recognized", "new_diagnosis_label"]).copy()
test_df["task4_Complex scene description_obraz u jezera_recognized"] = test_df["task4_Complex scene description_obraz u jezera_recognized"].astype(str)
test_df["new_diagnosis_label"] = test_df["new_diagnosis_label"].astype(int)

test_texts = test_df["task4_Complex scene description_obraz u jezera_recognized"].tolist()
test_labels = test_df["new_diagnosis_label"].values

print("Počet instancí v testovací množině:", len(test_texts))

# =========================================
# Tokenizace testovacích dat
# =========================================

print("\nTokenizuji testovací data...")
test_input_ids, test_attention_mask = tokenize_texts(
    test_texts, tokenizer, MAX_LENGTH
)

test_labels_tensor = torch.tensor(test_labels, dtype=torch.long)

test_dataset = TensorDataset(
    test_input_ids, test_attention_mask, test_labels_tensor
)
test_dataloader = DataLoader(
    test_dataset, batch_size=BATCH_SIZE, shuffle=False
)

# =========================================
# Predikce na testovací množině
# =========================================

print("\nPredikuji na testovacích datech...")
model.eval()
all_test_probs = []
all_test_labels = []

with torch.no_grad():
    for batch in tqdm(test_dataloader, desc="Test predikce"):
        batch_input_ids = batch[0].to(device)
        batch_attention_mask = batch[1].to(device)
        batch_labels = batch[2].to(device)

        outputs = model(
            input_ids=batch_input_ids,
            attention_mask=batch_attention_mask
        )

        logits = outputs.logits
        probs = torch.softmax(logits, dim=1)[:, 1]  # pravděpodobnost třídy 1

        all_test_probs.extend(probs.cpu().numpy())
        all_test_labels.extend(batch_labels.cpu().numpy())

all_test_probs = np.array(all_test_probs)
all_test_labels = np.array(all_test_labels)

# Binární predikce s prahem 0.5
test_preds = (all_test_probs >= 0.5).astype(int)

# =============================================
# Metriky na testovací množině (včetně ROC AUC)
# =============================================

test_acc = accuracy_score(all_test_labels, test_preds)
test_precision, test_recall, test_f1, _ = precision_recall_fscore_support(
    all_test_labels,
    test_preds,
    average="binary",
    zero_division=0
)

try:
    test_auc = roc_auc_score(all_test_labels, all_test_probs)
except ValueError:
    test_auc = float("nan")

print("\n===== METRIKY NA TESTOVACÍ MNOŽINĚ =====")
print(f"Test Accuracy:  {test_acc:.4f}")
print(f"Test Precision: {test_precision:.4f}")
print(f"Test Recall:    {test_recall:.4f}")
print(f"Test F1:        {test_f1:.4f}")
print(f"Test ROC AUC:   {test_auc:.4f}")

# =========================================
# Uložení predikcí do CSV
# =========================================

test_df["Pravdepodobnost"] = all_test_probs
test_df["Klasifikace_pred"] = test_preds  # aby se odlišilo od původní Klasifikace

test_df.to_csv(OUTPUT_TEST_CSV_PATH, index=False)
print(f"\nVýsledné predikce byly uloženy do souboru:\n{OUTPUT_TEST_CSV_PATH}")


Počet instancí v testovací množině: 61

Tokenizuji testovací data...

Predikuji na testovacích datech...


Test predikce:   0%|          | 0/8 [00:00<?, ?it/s]


===== METRIKY NA TESTOVACÍ MNOŽINĚ =====
Test Accuracy:  0.8033
Test Precision: 0.7692
Test Recall:    0.5263
Test F1:        0.6250
Test ROC AUC:   0.7419

Výsledné predikce byly uloženy do souboru:
./texttest_predicted.csv


In [ ]:
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(all_test_labels, test_preds)

print("\n===== MATICE ZÁMĚN =====")
print(cm)


===== MATICE ZÁMĚN =====
[[39  3]
 [ 9 10]]


In [ ]:
# model.save_pretrained("/")

In [ ]:
# tokenizer.save_pretrained("/")